<div style="text-align: center;">
    <h1>Évaluation Intermédiaire</h1>
    <h2>Python pour la Data Science</h2>
    <hr>
    <p><strong>Réalisé par:</strong></p>
    <p>Anicet Marius YABOYA<br>
    Cheikna Amala YATABARE</p>
    <p><em>Mars 2026</em></p>
</div>

## Librairies utilisées

In [ ]:
! pip install pandas -q
! pip install matplotlib -q
! pip install cartiflette -q
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from cartiflette import carti_download
from Fonction_plot import plot_surrepresentation
from Fonction_carto import carte_candidat

## Données du projet

In [ ]:
elections = pd.read_csv(
'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'  
)
elections.head()

In [ ]:
#Nombre de lignes et de colonnes
print(f"La base de données contient {elections.shape[0]} lignes et {elections.shape[1]} colonnes.")
#Liste des colonnes
print("Les colonnes de la base de données sont :")
for col in elections.columns:
    print(f"- {col}")
#Types de données
print("\nTypes de données :")
print(elections.dtypes)


# 1. Explorations générales

### Question 1: Création et mise à jour des variables

In [ ]:
# code_commune : on concatène le code département (2 chiffres) et le code commune (3 chiffres)
elections['code_commune'] = (
    elections['code_departement'].astype(str).str.zfill(2)
    + elections['code_commune'].astype(str).str.zfill(3)
)

# candidat : prénom + nom
elections['candidat'] = elections['prenom'].astype(str) + ' ' + elections['nom'].astype(str)

elections[['code_commune', 'libelle_commune', 'candidat', 'voix']].head()

### Question 2: Mise en forme


In [ ]:
# Exclusion des lignes non-candidats (abstentions, blancs, nuls)
non_candidats = ['NAN ABSTENTIONS', 'NAN BLANCS', 'NAN NULS']
masque_candidats = ~elections['candidat'].str.upper().isin(non_candidats)

candidats = elections[masque_candidats]['candidat'].nunique()

print(f"En 2022, il y avait {candidats} candidats à l'élection présidentielle.")
print("Liste des candidats :")
for candidat in sorted(elections[masque_candidats]['candidat'].dropna().unique()):
    print(candidat)

### Question 3: Scores nationaux de chaque candidat

In [ ]:
# Votes exprimés uniquement (on exclut les non-candidats)
elections_candidats = elections[masque_candidats].copy()

# Scores nationaux
score_national = elections_candidats.groupby('candidat', as_index=False)['voix'].sum()
total_exprime = score_national['voix'].sum()

score_national = (
    elections_candidats
    .groupby('candidat', as_index=False)['voix']
    .sum()
    .rename(columns={'voix': 'votes_national'})
    .sort_values('votes_national', ascending=False)
    .reset_index(drop=True)
)

score_national['score_national'] = score_national['votes_national'] / total_exprime * 100

# Affichage 
display_elections = score_national.copy()
display_elections['votes_national'] = display_elections['votes_national'].apply(lambda x: f"{x:,.0f}".replace(',', ' '))
display_elections['score_national'] = display_elections['score_national'].apply(lambda x: f"{x:.2f}%")
display_elections.index += 1  

print("\n Résultats du 1er tour aux élections présidentielles de 2022: \n")
display_elections

On constate que les 2 candidats arrivés en tête sont Emmanuel Macron et Marine Le Pen, avec respectivement 27.84% et 23.15% des voix exprimées. Ils se sont donc qualifiés pour le second tour.


# 2. Comparaison des scores départements aux moyennes nationales.

### Question 4: Création du dataframe nommé score_departements

In [ ]:
elelections_candidats =elections[masque_candidats].copy()

# Somme des voix par département et candidat
votes_dep = (
   elections_candidats
    .groupby(['code_departement', 'candidat'], as_index=False)['voix']
    .sum()
    .rename(columns={'voix': 'votes_departement'})
)

# Total des votes exprimés par département
total_dep = (
    votes_dep.groupby('code_departement', as_index=False)['votes_departement']
    .sum()
    .rename(columns={'votes_departement': 'total_dep'})
)

# Calcul du pourcentage par department
score_departements = votes_dep.merge(total_dep, on='code_departement')
score_departements['score_departement'] = score_departements['votes_departement'] / score_departements['total_dep'] * 100
score_departements = score_departements.sort_values(['code_departement', 'votes_departement'], ascending=[True, False])

# Vérification pour le département 11
aude = score_departements[score_departements['code_departement'] == '11'].copy()
aude['score_departement'] = aude['score_departement'].apply(lambda x: f"{x:.2f}%")
display(aude)

On constate que dans le département de l'Aude, Marine Le Pen est en tête avec un score de 30.14%, suivie d'Emmanuel Macron qui a 20.29%.

###  Question 5 – Jointure avec le niveau national pour comparaison

In [ ]:
score_departements = score_departements.merge(
    score_national[['candidat', 'votes_national', 'score_national']],
    on='candidat'
)

# Vérification pour l'Aude
print(" Vérification pour le département 11 (Aude) – 3 premiers candidats :")
check = score_departements[score_departements['code_departement'] == '11'].head(3).copy()
check['score_departement'] = check['score_departement'].apply(lambda x: f"{x:.2f}%")
check['score_national'] = check['score_national'].apply(lambda x: f"{x:.2f}%")
check

###  Question 6 : Création de la Variable `surrepresentation`

In [ ]:
# surrepresentation = (score_dep - score_national) / score_national * 100
score_departements['surrepresentation'] = (
    (score_departements['score_departement'] - score_departements['score_national'])
    / score_departements['score_national']
    * 100
)


print("Exemple (dept 11, Marine Le Pen) :",
      round(score_departements[
          (score_departements['code_departement'] == '11') &
          (score_departements['candidat'] == 'Marine LE PEN')
      ]['surrepresentation'].values[0], 2), "%")

score_departements.head()

###  Question 7 – Visualisation des surreprésentations

In [ ]:
# Test sur Éric Zemmour
plot_surrepresentation('Éric ZEMMOUR', score_departements, top_n=5)

In [ ]:
# Test sur Marine Le Pen
plot_surrepresentation('Marine LE PEN', score_departements, top_n=5)

In [ ]:
# Test sur Jean-Luc Mélenchon
plot_surrepresentation('Jean-Luc MÉLENCHON', score_departements, top_n=10)

## 3.  Cartographie

In [ ]:
# Téléchargement du fond de carte
departement_borders = carti_download(
    values=["France"],
    crs=4326,
    borders="DEPARTEMENT",
    vectorfile_format="geojson",
    simplification=50,
    filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2022
)

### Question 8: Fonction permettant de restreindre le score_departements par candidat

In [ ]:
carte_candidat('Marine LE PEN', score_departements, departement_borders)

In [ ]:
#test sur Éric Zemmour
carte_candidat('Éric ZEMMOUR', score_departements, departement_borders)


In [ ]:
#test sur Emmanuel Macron
carte_candidat('Emmanuel MACRON', score_departements, departement_borders)